# 02 · Avaliação do retrieval

Pergunta que este notebook responde: **qual estratégia de busca encontra o artigo certo com mais frequência?**

| Método | Ideia |
|---|---|
| BM25 | busca lexical clássica (palavras em comum) |
| E5 (multilingual-e5-base) | embeddings densos, modelo médio |
| BGE-M3 | embeddings densos, modelo maior e multilíngue |
| Híbrido (BM25 + BGE-M3) | combina as duas listas com *Reciprocal Rank Fusion* |
| Híbrido + reranker | o top-30 do híbrido é reordenado por um cross-encoder (bge-reranker-v2-m3) |

Métricas no nível de **artigo** (vários chunks do mesmo artigo contam como um acerto só):
Recall@k, MRR@10 e nDCG@10, com **intervalo de confiança de 95% por bootstrap** —
com algumas centenas de perguntas, diferenças de 1–2 pontos podem ser só ruído.

**Entrada:** adicione a saída do notebook 01 em *Add Input → Your Work → Notebooks*.
**Configuração:** GPU T4 x2, Internet On. ⏱️ ~15–20 min de GPU.

In [ ]:
!pip install -q rank_bm25

In [ ]:
import os, re, json, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

IN_KAGGLE = Path("/kaggle").exists()
OUT_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("saida_02")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"


def achar_arquivo(nome: str) -> Path:
    """Procura o arquivo nas entradas do Kaggle (o caminho depende do nome que você deu ao notebook 01)."""
    raizes = [Path("/kaggle/input")] if IN_KAGGLE else [Path("saida_01"), Path(".")]
    for raiz in raizes:
        achados = sorted(raiz.glob(f"**/{nome}"))
        if achados:
            return achados[0]
    raise FileNotFoundError(f"{nome} não encontrado. Adicionou a saída do notebook 01 como input?")


chunks = pd.read_parquet(achar_arquivo("chunks.parquet"))
avaliacao = pd.read_parquet(achar_arquivo("avaliacao.parquet"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

E5_ID = "intfloat/multilingual-e5-base"
BGE_ID = "BAAI/bge-m3"
RERANKER_ID = "BAAI/bge-reranker-v2-m3"
if SMOKE_TEST:  # modelos pequenos só para validar o código na CPU
    E5_ID = BGE_ID = "intfloat/multilingual-e5-small"
    RERANKER_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

TOP_K = 10           # quantos artigos cada método devolve
CANDIDATOS_RERANK = 30
RRF_K = 60           # constante padrão do Reciprocal Rank Fusion

# Só perguntas que TÊM resposta no corpus entram na avaliação de retrieval
q = avaliacao[avaliacao.artigo_id.notna()].reset_index(drop=True)
print(f"{len(chunks):,} chunks | {len(q)} perguntas com artigo esperado")
print(q.origem.value_counts())

## 1. BM25

Tokenização simples para português: minúsculas, sem acento, sem stopwords e com *stemming* RSLP
("consumidores" e "consumidor" viram o mesmo token).

In [ ]:
from rank_bm25 import BM25Okapi

try:
    import nltk
    nltk.download("stopwords", quiet=True)
    nltk.download("rslp", quiet=True)
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("portuguese"))
    _stemmer = nltk.stem.RSLPStemmer()
    stem = _stemmer.stem
except Exception as e:  # sem internet / sem nltk: segue sem stemming
    print("NLTK indisponível, BM25 sem stemming:", e)
    STOP = set("a o e de da do das dos em no na nos nas um uma para por com que se os as ao aos é ou".split())
    stem = lambda w: w


def sem_acento(t: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")


def tokenizar(t: str) -> list[str]:
    palavras = re.findall(r"\w+", t.lower())
    return [sem_acento(stem(w)) for w in palavras if w not in STOP and len(w) > 1]


t0 = time.time()
bm25 = BM25Okapi([tokenizar(t) for t in tqdm(chunks.texto_indexado, desc="BM25")])
print(f"Índice BM25: {time.time() - t0:.1f}s")


def buscar_bm25(pergunta: str, k: int) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenizar(pergunta))
    idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in idx]

## 2. Embeddings densos

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

# O E5 foi treinado com prefixos "query: " / "passage: "; sem eles a qualidade cai bastante.
PREFIXOS = {E5_ID: ("query: ", "passage: ")} if "e5" in E5_ID else {}


def indexar(model_id: str):
    modelo = SentenceTransformer(model_id, device=DEVICE)
    if DEVICE == "cuda":
        modelo.half()
    pq, pp = PREFIXOS.get(model_id, ("", ""))
    t0 = time.time()
    emb = modelo.encode([pp + t for t in chunks.texto_indexado], batch_size=32, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True)
    print(f"{model_id}: {emb.shape} em {time.time() - t0:.0f}s")
    return modelo, emb.astype(np.float32), pq


densos = {}
for nome, mid in [("e5", E5_ID), ("bge_m3", BGE_ID)]:
    modelo, emb, prefixo = indexar(mid)
    densos[nome] = {"modelo": modelo, "emb": emb, "prefixo": prefixo}

# Salva o índice BGE-M3 para o app (Hugging Face Space) não precisar recalcular
np.save(OUT_DIR / "emb_bge_m3.npy", densos["bge_m3"]["emb"].astype(np.float16))


def buscar_denso(nome: str, pergunta: str, k: int) -> list[tuple[int, float]]:
    d = densos[nome]
    qv = d["modelo"].encode([d["prefixo"] + pergunta], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = d["emb"] @ qv.astype(np.float32)
    idx = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]

## 3. Híbrido (RRF) e reranker

**RRF:** `score(d) = Σ 1 / (60 + posição_d)` em cada lista. Não precisa calibrar escalas de score
diferentes (BM25 vai de 0 a ~30; cosseno, de -1 a 1), por isso é o padrão em sistemas de produção.

In [ ]:
reranker = CrossEncoder(RERANKER_ID, device=DEVICE, max_length=512)
if DEVICE == "cuda":
    reranker.model.half()


def rrf(*listas: list[tuple[int, float]]) -> list[tuple[int, float]]:
    acumulado = {}
    for lista in listas:
        for pos, (i, _) in enumerate(lista):
            acumulado[i] = acumulado.get(i, 0.0) + 1.0 / (RRF_K + pos + 1)
    return sorted(acumulado.items(), key=lambda x: -x[1])


def buscar_hibrido(pergunta: str, k: int) -> list[tuple[int, float]]:
    return rrf(buscar_bm25(pergunta, CANDIDATOS_RERANK), buscar_denso("bge_m3", pergunta, CANDIDATOS_RERANK))[:k]


def buscar_hibrido_rerank(pergunta: str, k: int) -> list[tuple[int, float]]:
    candidatos = buscar_hibrido(pergunta, CANDIDATOS_RERANK)
    pares = [(pergunta, chunks.texto_indexado.iloc[i]) for i, _ in candidatos]
    scores = reranker.predict(pares, batch_size=32, show_progress_bar=False)
    ordem = np.argsort(-np.asarray(scores))
    return [(candidatos[j][0], float(scores[j])) for j in ordem[:k]]


METODOS = {
    "BM25": lambda p, k: buscar_bm25(p, k),
    "E5": lambda p, k: buscar_denso("e5", p, k),
    "BGE-M3": lambda p, k: buscar_denso("bge_m3", p, k),
    "Híbrido": buscar_hibrido,
    "Híbrido + reranker": buscar_hibrido_rerank,
}

## 4. Rodando todos os métodos

Buscamos mais chunks do que `TOP_K` e depois deduplicamos por artigo, para que cada método devolva
exatamente `TOP_K` **artigos** distintos.

In [ ]:
def para_artigos(resultado: list[tuple[int, float]], k: int) -> list[str]:
    vistos = []
    for i, _ in resultado:
        a = chunks.artigo_id.iloc[i]
        if a not in vistos:
            vistos.append(a)
        if len(vistos) == k:
            break
    return vistos


resultados, latencias = {}, {}
for nome, fn in METODOS.items():
    t0 = time.time()
    brutos = [fn(p, TOP_K * 3) for p in tqdm(q.pergunta, desc=nome)]
    latencias[nome] = (time.time() - t0) / len(q) * 1000
    resultados[nome] = {
        "artigos": [para_artigos(b, TOP_K) for b in brutos],
        "chunks": [[chunks.chunk_id.iloc[i] for i, _ in b[:TOP_K]] for b in brutos],
    }

## 5. Métricas com intervalo de confiança

In [ ]:
def posicao_acerto(previstos: list[str], esperado: str) -> int | None:
    return previstos.index(esperado) + 1 if esperado in previstos else None


def metricas_por_pergunta(previstos_lista, esperados) -> pd.DataFrame:
    linhas = []
    for prev, esp in zip(previstos_lista, esperados):
        pos = posicao_acerto(prev, esp)
        linhas.append({
            "Recall@1": float(pos == 1), "Recall@3": float(pos is not None and pos <= 3),
            "Recall@5": float(pos is not None and pos <= 5), "Recall@10": float(pos is not None),
            "MRR@10": 1.0 / pos if pos else 0.0,
            # Com um único artigo relevante, nDCG = 1/log2(1+posição)
            "nDCG@10": 1.0 / np.log2(pos + 1) if pos else 0.0,
        })
    return pd.DataFrame(linhas)


def bootstrap_ic(valores: np.ndarray, n: int = 2000, seed: int = 0) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    medias = rng.choice(valores, size=(n, len(valores)), replace=True).mean(axis=1)
    return float(np.percentile(medias, 2.5)), float(np.percentile(medias, 97.5))


por_pergunta = {nome: metricas_por_pergunta(r["artigos"], q.artigo_id) for nome, r in resultados.items()}

linhas = []
for nome, df in por_pergunta.items():
    linha = {"Método": nome}
    for col in df.columns:
        linha[col] = df[col].mean()
    lo, hi = bootstrap_ic(df["Recall@5"].to_numpy())
    linha["Recall@5 IC95%"] = f"[{lo:.3f}, {hi:.3f}]"
    linha["Latência (ms/consulta)"] = latencias[nome]
    linhas.append(linha)
tabela = pd.DataFrame(linhas).set_index("Método")
tabela.round(3)

### A diferença entre o melhor e o segundo melhor é real?
Teste pareado por bootstrap: reamostramos as *mesmas* perguntas para os dois métodos.

In [ ]:
def bootstrap_pareado(a: np.ndarray, b: np.ndarray, n: int = 5000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    difs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    return {"diferença": float(a.mean() - b.mean()), "IC95%": (float(np.percentile(difs, 2.5)),
            float(np.percentile(difs, 97.5))), "p(a<=b)": float((difs <= 0).mean())}


ranking = tabela["MRR@10"].sort_values(ascending=False).index.tolist()
melhor, segundo = ranking[0], ranking[1]
comparacao = bootstrap_pareado(por_pergunta[melhor]["MRR@10"].to_numpy(), por_pergunta[segundo]["MRR@10"].to_numpy())
print(f"MRR@10: {melhor} vs {segundo} → {comparacao}")
comparacao_bm25 = bootstrap_pareado(por_pergunta[melhor]["MRR@10"].to_numpy(), por_pergunta["BM25"]["MRR@10"].to_numpy())
print(f"MRR@10: {melhor} vs BM25 → {comparacao_bm25}")

### Recortes: por origem da pergunta e por lei
É aqui que aparecem os insights: perguntas golden (linguagem real) costumam ser bem mais difíceis
que as sintéticas, e leis longas como a CLT têm mais "artigos parecidos" competindo entre si.

In [ ]:
recortes = []
for nome, df in por_pergunta.items():
    tmp = df.assign(origem=q.origem.values, lei=q.lei.values, metodo=nome)
    recortes.append(tmp)
recortes = pd.concat(recortes)
print("Recall@5 por origem:")
print(recortes.pivot_table(index="metodo", columns="origem", values="Recall@5").reindex(METODOS).round(3))
print("\nRecall@5 por lei:")
print(recortes.pivot_table(index="metodo", columns="lei", values="Recall@5").reindex(METODOS).round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ks = [1, 3, 5, 10]
for nome, df in por_pergunta.items():
    ax.plot(ks, [df[f"Recall@{k}"].mean() for k in ks], marker="o", label=nome)
ax.set_xlabel("k (artigos recuperados)")
ax.set_ylabel("Recall@k")
ax.set_xticks(ks)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Recall@k por método de retrieval")
fig.tight_layout()
fig.savefig(OUT_DIR / "recall_at_k.png", dpi=150)
plt.show()

### Análise de erros
Sempre olhe os casos em que o **melhor** método errou. Normalmente são: pergunta ambígua,
mais de um artigo que responde (o "erro" é do dataset), ou vocabulário muito diferente da lei.

In [ ]:
erros = q.assign(top3=[r[:3] for r in resultados[melhor]["artigos"]],
                 acertou=por_pergunta[melhor]["Recall@10"].values)
for r in erros[erros.acertou == 0].head(8).itertuples():
    print(f"[{r.origem}] {r.pergunta}\n   esperado: {r.artigo_id} | top-3: {r.top3}\n")

## 6. Salvando para o notebook 03 e para o app

In [ ]:
recuperados = []
for nome, r in resultados.items():
    for qid, arts, chs in zip(q.qid, r["artigos"], r["chunks"]):
        recuperados.append({"qid": qid, "metodo": nome, "artigos": arts, "chunks": chs})

# Perguntas fora do escopo também precisam de contexto recuperado para o notebook 03
fora = avaliacao[avaliacao.artigo_id.isna()]
for nome, fn in METODOS.items():
    for qid, p in zip(fora.qid, fora.pergunta):
        b = fn(p, TOP_K * 3)
        recuperados.append({"qid": qid, "metodo": nome, "artigos": para_artigos(b, TOP_K),
                            "chunks": [chunks.chunk_id.iloc[i] for i, _ in b[:TOP_K]]})

pd.DataFrame(recuperados).to_parquet(OUT_DIR / "recuperados.parquet", index=False)
tabela.to_csv(OUT_DIR / "metricas_retrieval.csv")
chunks.to_parquet(OUT_DIR / "chunks.parquet", index=False)
avaliacao.to_parquet(OUT_DIR / "avaliacao.parquet", index=False)
json.dump({"melhor_metodo": melhor, "comparacao_vs_segundo": comparacao, "comparacao_vs_bm25": comparacao_bm25,
           "tabela": tabela.reset_index().to_dict(orient="records")},
          open(OUT_DIR / "resumo_retrieval.json", "w"), ensure_ascii=False, indent=2, default=str)
print("Melhor método:", melhor)
print("Arquivos:", sorted(p.name for p in OUT_DIR.iterdir()))